In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

torch.manual_seed(42)

# 1. Data loading
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.5, 0.5, 0.5),
        (0.5, 0.5, 0.5)
    )
])

batch_size = 50

trainset = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

trainloader = torch.utils.data.DataLoader(
    trainset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0
)

testset = torchvision.datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

testloader = torch.utils.data.DataLoader(
    testset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

classes = (
    "plane", "car", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
)

# 2. Define network parameters
Din = 3 * 32 * 32  # Input size
H = 100           # Hidden-layer size
K = 10            # Output size

std = 0.01

# Input -> hidden layer
w1 = torch.randn(Din, H) * std
b1 = torch.zeros(H)

# Hidden -> output layer
w2 = torch.randn(H, K) * std
b2 = torch.zeros(K)

# Hyperparameters
iterations = 10  # Number of epochs
lr = 0.1
lr_decay = 0.9
reg = 0.0

loss_history = []

# 3. Training loop
for t in range(iterations):
    running_loss = 0.0

    for i, data in enumerate(trainloader, 0):
        # Get inputs and labels
        inputs, labels = data

        Ntr = inputs.shape[0]
        x_train = inputs.view(Ntr, -1)

        y_train_onehot = nn.functional.one_hot(
            labels, K
        ).float()

        z1 = x_train.mm(w1) + b1
        hidden = torch.sigmoid(z1)

        y_pred = hidden.mm(w2) + b2

        log_probs = y_pred - torch.logsumexp(
            y_pred,
            dim=1,
            keepdim=True
        )
        probs = torch.exp(log_probs)

        loss = (
            -(1.0 / Ntr) * torch.sum(y_train_onehot * log_probs)
            + reg * (
                torch.sum(w1**2) + torch.sum(w2**2)
            )
        )

        loss_history.append(loss.item())
        running_loss += loss.item()


        # Derivative of softmax + cross-entropy
        dy_pred = (probs - y_train_onehot) / Ntr

        # Output-layer gradients
        dw2 = hidden.t().mm(dy_pred) + 2.0 * reg * w2
        db2 = dy_pred.sum(dim=0)

        # Gradient through hidden layer
        dhidden = dy_pred.mm(w2.t())

        # Sigmoid derivative
        dz1 = dhidden * hidden * (1.0 - hidden)

        # Input-to-hidden layer gradients
        dw1 = x_train.t().mm(dz1) + 2.0 * reg * w1
        db1 = dz1.sum(dim=0)

        w1 -= lr * dw1
        b1 -= lr * db1

        w2 -= lr * dw2
        b2 -= lr * db2

    # Print mean loss for each epoch
    print(
        f"Epoch {t + 1}/{iterations}, "
        f"Loss: {running_loss / len(trainloader):.4f}"
    )

    # Learning-rate decay
    lr *= lr_decay

# 4. Plotting the loss history
plt.plot(loss_history)
plt.title("Loss History")
plt.xlabel("Iteration (mini-batch)")
plt.ylabel("Cross-Entropy Loss")
plt.show()

# 5. Calculate accuracy on training set
correct_train = 0
total_train = 0

with torch.no_grad():
    for data in trainloader:
        inputs, labels = data

        Ntr = inputs.shape[0]
        x_train = inputs.view(Ntr, -1)

        # Forward pass
        hidden_train = torch.sigmoid(x_train.mm(w1) + b1)
        y_train_pred = hidden_train.mm(w2) + b2

        predicted_train = torch.argmax(y_train_pred, dim=1)

        total_train += labels.size(0)
        correct_train += (
            predicted_train == labels
        ).sum().item()

train_acc = 100.0 * correct_train / total_train

print(f"Training accuracy: {train_acc:.2f}%")

# 6. Calculate accuracy on test set
correct_test = 0
total_test = 0

with torch.no_grad():
    for data in testloader:
        inputs, labels = data

        Nte = inputs.shape[0]
        x_test = inputs.view(Nte, -1)

        # Forward pass
        hidden_test = torch.sigmoid(x_test.mm(w1) + b1)
        y_test_pred = hidden_test.mm(w2) + b2

        predicted_test = torch.argmax(y_test_pred, dim=1)

        total_test += labels.size(0)
        correct_test += (
            predicted_test == labels
        ).sum().item()

test_acc = 100.0 * correct_test / total_test

print(f"Test accuracy: {test_acc:.2f}%")